# HHEM-2.1-Open vs. Current RAGAS Faithfulness Setup

Evaluates whether Vectara's HHEM-2.1-Open classifier is a viable replacement/hybrid for our LiteLLM faithfulness guardrail.

Compares three scoring architectures on the same labeled test cases:

| # | Architecture | LLM calls |
|---|---|---|
| 1 | Standalone HHEM (whole answer vs. context) | 0 |
| 2 | `FaithfulnesswithHHEM` (LLM decomposes claims, HHEM verifies) | 1 |
| 3 | Current prod `Faithfulness` (LLM decomposes, LLM verifies) | 2 |

**Runtime:** Colab, GPU (T4) recommended for HHEM inference. `Runtime > Change runtime type > T4 GPU`.

Reuses the same judge model as production (`groq/openai/gpt-oss-20b` via Groq) for the LLM-calling scorers, so results are directly comparable to the 0.7 threshold already in use.

## 1. Setup

Installs the libraries we need and confirms a GPU is attached.

- `transformers` + `torch` — to load HHEM-2.1-Open directly from Hugging Face for the standalone scorer.
- `ragas` — provides both `Faithfulness` (current prod scorer) and `FaithfulnesswithHHEM` (hybrid scorer) so we don't reimplement either.
- `openai` — the client `ragas.llms.llm_factory` wraps to talk to the judge model through Groq's OpenAI-compatible endpoint.

The GPU check just prints what Colab gave us — if it prints `cpu`, go to `Runtime > Change runtime type` and pick a GPU before continuing, since HHEM will be noticeably slower on CPU.

In [ ]:
%pip install -q transformers torch ragas openai

import torch
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run this cell.")